# Notebook 04 — Sales Prediction

**Retail Sales AI Analytics — IBM SkillsBuild Capstone**

This notebook covers:
- Monthly sales aggregation
- Time-based train/test split
- Naive baseline, Linear Regression, Random Forest evaluation
- MAE, RMSE, R² metrics
- Forecast vs actual visualisation
- Explicit discussion of model limitations

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)

from src.feature_engineering import load_processed
from src.forecasting import (
    prepare_monthly_series, train_test_split_time, run_forecasting
)

In [ ]:
df = load_processed()
print(f'Loaded {len(df):,} rows')

In [ ]:
# Prepare monthly series
monthly = prepare_monthly_series(df)
print(f'Monthly data points: {len(monthly)}')
monthly.head(10)

In [ ]:
# Train/test split (last 12 months = test)
train, test = train_test_split_time(monthly, test_months=12)
print(f'Train: {len(train)} months | Test: {len(test)} months')
print(f'Train range: {train["Year_Month"].iloc[0]} to {train["Year_Month"].iloc[-1]}')
print(f'Test range:  {test["Year_Month"].iloc[0]} to {test["Year_Month"].iloc[-1]}')

In [ ]:
# Full forecasting pipeline
results = run_forecasting(df)
metrics_df = results['metrics_df']
print('\nModel Metrics:')
print(metrics_df.to_string(index=False))

In [ ]:
# Plot forecast
train_r = results['train']
test_r  = results['test']

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train_r['Year_Month'], train_r['Sales']/1e3, label='Train Actual', color='steelblue', lw=2)
ax.plot(test_r['Year_Month'],  test_r['Sales']/1e3,  label='Test Actual', color='steelblue', lw=2, ls='--')
ax.plot(test_r['Year_Month'],  test_r['Pred_LR']/1e3, label='Linear Regression', color='darkorange', lw=1.8)
ax.plot(test_r['Year_Month'],  test_r['Pred_RF']/1e3, label='Random Forest', color='green', lw=1.8)
ax.plot(test_r['Year_Month'],  test_r['Pred_Naive']/1e3, label='Naive Baseline', color='gray', lw=1.2, ls=':')

all_months = list(train_r['Year_Month']) + list(test_r['Year_Month'])
n = len(all_months)
step = max(1, n//12)
ax.set_xticks(range(0, n, step))
ax.set_xticklabels(all_months[::step], rotation=45, ha='right', fontsize=8)
ax.axvline(x=len(train_r)-1, color='gray', linestyle=':', lw=1, label='Train/Test split')
ax.set_title('Monthly Sales: Actual vs Predicted')
ax.set_ylabel('Sales ($k)')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x:.0f}k'))
plt.tight_layout()
plt.show()

## Model Limitations

- The dataset spans only ~4 years, giving limited monthly data points.
- Only a numeric time index is used as a feature (linear trend).
- No explicit seasonality features are included to avoid overfitting.
- Forecasts beyond the observed date range should not be treated as business projections.
- The Naive Baseline is included as a sanity check — a model that cannot beat the naive baseline is not useful.